In [ ]:
import numpy as np
import pandas as pd
import scipy.io as sio
import os, glob, warnings, time
from scipy import stats
from scipy.signal import welch
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, optimizers
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb

warnings.filterwarnings('ignore')
tf.random.set_seed(42)
np.random.seed(42)
os.makedirs('/kaggle/working/figures', exist_ok=True)
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

In [ ]:
BASE = '/kaggle/input/datasets/emperorpein/mfpt-fault-datasets/MFPT Fault Data Sets'

# Hardcode every .mat file with its severity label
# Severity: 0 = healthy, 1 = outer race fault, 2 = inner race fault
file_list = [
    # Baseline (healthy)
    (f"{BASE}/1 - Three Baseline Conditions/baseline_1.mat", 0),
    (f"{BASE}/1 - Three Baseline Conditions/baseline_2.mat", 0),
    (f"{BASE}/1 - Three Baseline Conditions/baseline_3.mat", 0),
    # Outer race faults
    (f"{BASE}/2 - Three Outer Race Fault Conditions/OuterRaceFault_1.mat", 1),
    (f"{BASE}/2 - Three Outer Race Fault Conditions/OuterRaceFault_2.mat", 1),
    (f"{BASE}/2 - Three Outer Race Fault Conditions/OuterRaceFault_3.mat", 1),
    (f"{BASE}/3 - Seven More Outer Race Fault Conditions/OuterRaceFault_vload_1.mat", 1),
    (f"{BASE}/3 - Seven More Outer Race Fault Conditions/OuterRaceFault_vload_2.mat", 1),
    (f"{BASE}/3 - Seven More Outer Race Fault Conditions/OuterRaceFault_vload_3.mat", 1),
    (f"{BASE}/3 - Seven More Outer Race Fault Conditions/OuterRaceFault_vload_4.mat", 1),
    (f"{BASE}/3 - Seven More Outer Race Fault Conditions/OuterRaceFault_vload_5.mat", 1),
    (f"{BASE}/3 - Seven More Outer Race Fault Conditions/OuterRaceFault_vload_6.mat", 1),
    (f"{BASE}/3 - Seven More Outer Race Fault Conditions/OuterRaceFault_vload_7.mat", 1),
    # Inner race faults
    (f"{BASE}/4 - Seven Inner Race Fault Conditions/InnerRaceFault_vload_1.mat", 2),
    (f"{BASE}/4 - Seven Inner Race Fault Conditions/InnerRaceFault_vload_2.mat", 2),
    (f"{BASE}/4 - Seven Inner Race Fault Conditions/InnerRaceFault_vload_3.mat", 2),
    (f"{BASE}/4 - Seven Inner Race Fault Conditions/InnerRaceFault_vload_4.mat", 2),
    (f"{BASE}/4 - Seven Inner Race Fault Conditions/InnerRaceFault_vload_5.mat", 2),
    (f"{BASE}/4 - Seven Inner Race Fault Conditions/InnerRaceFault_vload_6.mat", 2),
    (f"{BASE}/4 - Seven Inner Race Fault Conditions/InnerRaceFault_vload_7.mat", 2),
    (f"{BASE}/5 - Analyses/InnerRaceFault_vload_7.mat", 2),
]

recordings = []
for filepath, severity in file_list:
    if not os.path.exists(filepath):
        print(f"  NOT FOUND: {filepath}")
        continue
    try:
        mat = sio.loadmat(filepath)
        b = mat['bearing']
        sig = b['gs'][0][0].flatten().astype(np.float64)
        sr = int(b['sr'][0][0].flatten()[0])
        load = float(b['load'][0][0].flatten()[0])
        recordings.append({
            'signal': sig,
            'severity': severity,
            'load': load,
            'name': os.path.basename(filepath),
            'sr': sr
        })
        print(f"  Loaded: {os.path.basename(filepath):40s} severity={severity}  load={load:.0f}  sr={sr}  samples={len(sig)}")
    except Exception as e:
        print(f"  ERROR: {os.path.basename(filepath)}: {e}")

print(f"\nTotal recordings loaded: {len(recordings)}")
print(f"  Healthy (0): {sum(1 for r in recordings if r['severity']==0)}")
print(f"  Outer race (1): {sum(1 for r in recordings if r['severity']==1)}")
print(f"  Inner race (2): {sum(1 for r in recordings if r['severity']==2)}")

In [ ]:
# MFPT has no run-to-failure data. RUL is constructed by ordering
# recordings from healthy to most degraded and assigning linear RUL.
# acknowledge as a limitation in the report.
RUL_CAP = 125
WINDOW_SIZE = 1024
STEP = 512

# Sorting: healthy first, then outer race, then inner race, higher load = more degraded within group
recordings.sort(key=lambda x: (x['severity'], x['load']))

all_windows = []
window_labels = []  # tracking which recording each window came from
for idx, rec in enumerate(recordings):
    sig = rec['signal']
    n_wins = (len(sig) - WINDOW_SIZE) // STEP + 1
    for i in range(n_wins):
        start = i * STEP
        all_windows.append(sig[start:start + WINDOW_SIZE])
    window_labels.extend([idx] * n_wins)
    print(f"  {rec['name']:40s} → {n_wins} windows")

all_windows = np.array(all_windows, dtype=np.float32)
N_WINDOWS = len(all_windows)
rul_labels = np.linspace(RUL_CAP, 0, N_WINDOWS).astype(np.float32)

print(f"\nTotal windows: {N_WINDOWS}")
print(f"RUL range: {rul_labels.max():.1f} → {rul_labels.min():.1f}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(rul_labels, lw=0.8)
ax.set_xlabel('Window Index'); ax.set_ylabel('RUL')
ax.set_title('MFPT Synthetic RUL (Healthy → Degraded)')
plt.savefig('/kaggle/working/figures/mfpt_rul_distribution.png', bbox_inches='tight')
plt.show()

In [ ]:
def extract_features(windows):
    feats = []
    for w in windows:
        rms = np.sqrt(np.mean(w ** 2))
        peak = np.max(np.abs(w))
        kurt = stats.kurtosis(w)
        skew = stats.skew(w)
        crest = peak / (rms + 1e-8)
        std = np.std(w)
        mean_abs = np.mean(np.abs(w))
        shape = rms / (mean_abs + 1e-8)
        freqs, psd = welch(w, nperseg=256)
        psd_norm = psd / (psd.sum() + 1e-8)
        spec_centroid = np.sum(freqs * psd_norm)
        spec_rms = np.sqrt(np.sum(freqs ** 2 * psd_norm))
        spec_kurt = stats.kurtosis(psd)
        peak_freq = freqs[np.argmax(psd)]
        feats.append([rms, peak, kurt, skew, crest, std, mean_abs, shape,
                      spec_centroid, spec_rms, spec_kurt, peak_freq])
    return np.array(feats, dtype=np.float32)

feature_names = ['RMS', 'Peak', 'Kurtosis', 'Skewness', 'CrestFactor',
                 'Std', 'MeanAbs', 'ShapeFactor',
                 'SpectralCentroid', 'SpectralRMS', 'SpectralKurtosis', 'PeakFreq']

print("Extracting features...")
X_feat = extract_features(all_windows)
n_tab_features = len(feature_names)
print(f"Feature matrix: {X_feat.shape}")

In [ ]:
# Temporal split - no shuffle - preserves degradation ordering
split = int(0.8 * N_WINDOWS)

X_feat_tr, X_feat_te = X_feat[:split], X_feat[split:]
X_seq_tr, X_seq_te = all_windows[:split], all_windows[split:]
y_tr, y_te = rul_labels[:split], rul_labels[split:]

# Scale tabular features
scaler = StandardScaler()
X_feat_tr_sc = scaler.fit_transform(X_feat_tr)
X_feat_te_sc = scaler.transform(X_feat_te)

# Normalise raw windows for DL
def norm_win(w):
    m = w.mean(axis=1, keepdims=True)
    s = w.std(axis=1, keepdims=True) + 1e-8
    return (w - m) / s

X_seq_tr_n = norm_win(X_seq_tr)[..., np.newaxis]  # add channel dim for Keras
X_seq_te_n = norm_win(X_seq_te)[..., np.newaxis]

print(f"Train: {len(y_tr)} windows | Test: {len(y_te)} windows")
print(f"Tabular: {X_feat_tr_sc.shape} | Sequences: {X_seq_tr_n.shape}")

In [ ]:
corrs = {}
for i, name in enumerate(feature_names):
    corrs[name] = np.corrcoef(X_feat_tr_sc[:, i], y_tr)[0, 1]
corrs = pd.Series(corrs).sort_values(key=abs, ascending=False)
print("MFPT Feature correlations with RUL:")
print(corrs.to_string())

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#2166ac' if v > 0 else '#b2182b' for v in corrs.values]
ax.barh(range(len(corrs)), abs(corrs.values), color=colors)
ax.set_yticks(range(len(corrs))); ax.set_yticklabels(corrs.index)
ax.set_xlabel('|Correlation with RUL|'); ax.invert_yaxis()
ax.set_title('MFPT: Feature Correlation with RUL')
plt.savefig('/kaggle/working/figures/mfpt_feature_corr.png', bbox_inches='tight')
plt.show()

time_feats = ['RMS', 'Peak', 'Kurtosis', 'Skewness', 'CrestFactor', 'Std', 'MeanAbs', 'ShapeFactor']
freq_feats = ['SpectralCentroid', 'SpectralRMS', 'SpectralKurtosis', 'PeakFreq']
print(f"\nMean |r|:  Time-domain={np.mean([abs(corrs[f]) for f in time_feats]):.4f}")
print(f"           Freq-domain={np.mean([abs(corrs[f]) for f in freq_feats]):.4f}")

In [ ]:
t0 = time.time()
lr_model = LinearRegression().fit(X_feat_tr_sc, y_tr)
lr_time = time.time() - t0

y_pred_lr = lr_model.predict(X_feat_te_sc)
lr_rmse = np.sqrt(mean_squared_error(y_te, y_pred_lr))
lr_mae = mean_absolute_error(y_te, y_pred_lr)
lr_r2 = r2_score(y_te, y_pred_lr)
print(f"LR: RMSE={lr_rmse:.2f}, MAE={lr_mae:.2f}, R²={lr_r2:.4f}, Time={lr_time:.3f}s")

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(y_te, y_pred_lr, alpha=0.3, s=10)
ax.plot([0, RUL_CAP], [0, RUL_CAP], 'r--')
ax.set_xlabel('Actual RUL'); ax.set_ylabel('Predicted RUL')
ax.set_title(f'MFPT Linear Regression: RMSE={lr_rmse:.2f}, R²={lr_r2:.4f}')
plt.savefig('/kaggle/working/figures/mfpt_lr_scatter.png')
plt.show()

In [ ]:
print("Grid search:")
grid_results = []
for md in [4, 6, 8]:
    for lr in [0.05, 0.1, 0.2]:
        m = xgb.XGBRegressor(n_estimators=400, learning_rate=lr, max_depth=md,
            subsample=0.8, colsample_bytree=0.8, random_state=42,
            early_stopping_rounds=20, eval_metric='rmse', verbosity=0)
        m.fit(X_feat_tr_sc, y_tr, eval_set=[(X_feat_te_sc, y_te)], verbose=0)
        pred = m.predict(X_feat_te_sc)
        rmse = np.sqrt(mean_squared_error(y_te, pred))
        grid_results.append({'depth': md, 'lr': lr, 'rmse': rmse})
        print(f"  depth={md}, lr={lr}: RMSE={rmse:.2f}")

gdf = pd.DataFrame(grid_results)
best = gdf.loc[gdf['rmse'].idxmin()]
print(f"Best: depth={int(best['depth'])}, lr={best['lr']}, RMSE={best['rmse']:.2f}")

t0 = time.time()
xgb_model = xgb.XGBRegressor(n_estimators=400, learning_rate=best['lr'],
    max_depth=int(best['depth']), subsample=0.8, colsample_bytree=0.8,
    random_state=42, early_stopping_rounds=20, eval_metric='rmse', verbosity=0)
xgb_model.fit(X_feat_tr_sc, y_tr,
              eval_set=[(X_feat_tr_sc, y_tr), (X_feat_te_sc, y_te)], verbose=0)
xgb_time = time.time() - t0

y_pred_xgb = xgb_model.predict(X_feat_te_sc)
xgb_rmse = np.sqrt(mean_squared_error(y_te, y_pred_xgb))
xgb_mae = mean_absolute_error(y_te, y_pred_xgb)
xgb_r2 = r2_score(y_te, y_pred_xgb)
print(f"\nXGBoost: RMSE={xgb_rmse:.2f}, MAE={xgb_mae:.2f}, R²={xgb_r2:.4f}")
print(f"vs LR: {(1-xgb_rmse/lr_rmse)*100:+.1f}%")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].scatter(y_te, y_pred_xgb, alpha=0.3, s=10)
axes[0].plot([0,RUL_CAP],[0,RUL_CAP],'r--')
axes[0].set_title(f'MFPT XGBoost: RMSE={xgb_rmse:.2f}, R²={xgb_r2:.4f}')
ev = xgb_model.evals_result()
axes[1].plot(ev['validation_0']['rmse'], label='Train')
axes[1].plot(ev['validation_1']['rmse'], label='Test')
axes[1].legend(); axes[1].set_title('Learning Curves')
imp = xgb_model.feature_importances_
idx = np.argsort(imp)
axes[2].barh(range(len(idx)), imp[idx])
axes[2].set_yticks(range(len(idx))); axes[2].set_yticklabels([feature_names[i] for i in idx])
axes[2].set_title('Feature Importance')
plt.tight_layout()
plt.savefig('/kaggle/working/figures/mfpt_xgboost_results.png')
plt.show()

In [ ]:
cnn = keras.Sequential([
    layers.Input(shape=(WINDOW_SIZE, 1)),
    layers.Conv1D(64, 16, activation='relu', padding='same'),
    layers.BatchNormalization(), layers.MaxPooling1D(4), layers.Dropout(0.2),
    layers.Conv1D(128, 8, activation='relu', padding='same'),
    layers.BatchNormalization(), layers.MaxPooling1D(4), layers.Dropout(0.2),
    layers.Conv1D(64, 4, activation='relu', padding='same'),
    layers.BatchNormalization(), layers.GlobalAveragePooling1D(), layers.Dropout(0.3),
    layers.Dense(64, activation='relu'), layers.Dropout(0.2),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='linear')
])
cnn.compile(optimizer=optimizers.Adam(1e-3), loss='mse', metrics=['mae'])
cnn.summary()

t0 = time.time()
h_cnn = cnn.fit(X_seq_tr_n, y_tr, validation_data=(X_seq_te_n, y_te),
                epochs=60, batch_size=64,
                callbacks=[callbacks.EarlyStopping('val_loss', patience=10, restore_best_weights=True),
                           callbacks.ReduceLROnPlateau('val_loss', factor=0.5, patience=5)],
                verbose=1)
cnn_time = time.time() - t0

y_pred_cnn = cnn.predict(X_seq_te_n, batch_size=64).ravel()
cnn_rmse = np.sqrt(mean_squared_error(y_te, y_pred_cnn))
cnn_mae = mean_absolute_error(y_te, y_pred_cnn)
cnn_r2 = r2_score(y_te, y_pred_cnn)

print(f"\n1D-CNN: RMSE={cnn_rmse:.2f}, MAE={cnn_mae:.2f}, R²={cnn_r2:.4f}")
print(f"Params={cnn.count_params():,}, Time={cnn_time:.0f}s, Epochs={len(h_cnn.history['loss'])}")
print(f"vs XGBoost: {(1-cnn_rmse/xgb_rmse)*100:+.1f}%")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(h_cnn.history['loss'], label='Train'); axes[0].plot(h_cnn.history['val_loss'], label='Val')
axes[0].legend(); axes[0].set_yscale('log'); axes[0].set_title('MFPT 1D-CNN Loss')
axes[1].scatter(y_te, y_pred_cnn, alpha=0.3, s=10, c='darkorange')
axes[1].plot([0,RUL_CAP],[0,RUL_CAP],'r--')
axes[1].set_title(f'MFPT 1D-CNN: RMSE={cnn_rmse:.2f}, R²={cnn_r2:.4f}')
r = y_te - y_pred_cnn
axes[2].hist(r, bins=50, color='darkorange'); axes[2].axvline(0, color='red', ls='--')
axes[2].set_title(f'Residuals: mean={r.mean():.2f}, std={r.std():.2f}')
plt.tight_layout()
plt.savefig('/kaggle/working/figures/mfpt_cnn_results.png')
plt.show()

In [ ]:
cl = keras.Sequential([
    layers.Input(shape=(WINDOW_SIZE, 1)),
    layers.Conv1D(64, 16, activation='relu', padding='same'),
    layers.BatchNormalization(), layers.MaxPooling1D(4), layers.Dropout(0.2),
    layers.Conv1D(128, 8, activation='relu', padding='same'),
    layers.BatchNormalization(), layers.MaxPooling1D(4), layers.Dropout(0.2),
    layers.LSTM(64, return_sequences=True), layers.Dropout(0.2),
    layers.LSTM(32, return_sequences=False), layers.Dropout(0.2),
    layers.Dense(32, activation='relu'), layers.Dropout(0.2),
    layers.Dense(1, activation='linear')
])
cl.compile(optimizer=optimizers.Adam(1e-3), loss='mse', metrics=['mae'])
cl.summary()

t0 = time.time()
h_cl = cl.fit(X_seq_tr_n, y_tr, validation_data=(X_seq_te_n, y_te),
              epochs=60, batch_size=64,
              callbacks=[callbacks.EarlyStopping('val_loss', patience=10, restore_best_weights=True),
                         callbacks.ReduceLROnPlateau('val_loss', factor=0.5, patience=5)],
              verbose=1)
cl_time = time.time() - t0

y_pred_cl = cl.predict(X_seq_te_n, batch_size=64).ravel()
cl_rmse = np.sqrt(mean_squared_error(y_te, y_pred_cl))
cl_mae = mean_absolute_error(y_te, y_pred_cl)
cl_r2 = r2_score(y_te, y_pred_cl)

print(f"\nCNN-LSTM: RMSE={cl_rmse:.2f}, MAE={cl_mae:.2f}, R²={cl_r2:.4f}")
print(f"Params={cl.count_params():,}, Time={cl_time:.0f}s, Epochs={len(h_cl.history['loss'])}")
print(f"vs XGBoost: {(1-cl_rmse/xgb_rmse)*100:+.1f}%")
print(f"vs CNN:     {(1-cl_rmse/cnn_rmse)*100:+.1f}%")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(h_cl.history['loss'], label='Train'); axes[0].plot(h_cl.history['val_loss'], label='Val')
axes[0].legend(); axes[0].set_yscale('log'); axes[0].set_title('MFPT CNN-LSTM Loss')
axes[1].scatter(y_te, y_pred_cl, alpha=0.3, s=10, c='forestgreen')
axes[1].plot([0,RUL_CAP],[0,RUL_CAP],'r--')
axes[1].set_title(f'MFPT CNN-LSTM: RMSE={cl_rmse:.2f}, R²={cl_r2:.4f}')
r2 = y_te - y_pred_cl
axes[2].hist(r2, bins=50, color='forestgreen'); axes[2].axvline(0, color='red', ls='--')
axes[2].set_title(f'Residuals: mean={r2.mean():.2f}, std={r2.std():.2f}')
plt.tight_layout()
plt.savefig('/kaggle/working/figures/mfpt_cnn_lstm_results.png')
plt.show()

In [ ]:
print("\n=== MFPT MODEL COMPARISON ===")
print(f"{'Model':<20} {'RMSE':>8} {'MAE':>8} {'R²':>8} {'Time':>10}")
print("-" * 58)
for name, rmse, mae, r2, t in [
    ('Linear Regression', lr_rmse, lr_mae, lr_r2, lr_time),
    ('XGBoost', xgb_rmse, xgb_mae, xgb_r2, xgb_time),
    ('1D-CNN', cnn_rmse, cnn_mae, cnn_r2, cnn_time),
    ('CNN-LSTM', cl_rmse, cl_mae, cl_r2, cl_time)]:
    print(f"{name:<20} {rmse:>8.2f} {mae:>8.2f} {r2:>8.4f} {t:>9.1f}s")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
names = ['LR', 'XGB', 'CNN', 'CNN-LSTM']
c = ['grey', 'steelblue', 'darkorange', 'forestgreen']
rmses = [lr_rmse, xgb_rmse, cnn_rmse, cl_rmse]
axes[0].bar(names, rmses, color=c); axes[0].set_ylabel('RMSE'); axes[0].set_title('MFPT: RMSE')
for i,v in enumerate(rmses): axes[0].text(i, v+0.3, f'{v:.2f}', ha='center', fontsize=9)
axes[1].bar(names, [lr_r2, xgb_r2, cnn_r2, cl_r2], color=c); axes[1].set_ylabel('R²'); axes[1].set_title('MFPT: R²')
axes[2].bar(names, [lr_time, xgb_time, cnn_time, cl_time], color=c)
axes[2].set_ylabel('Seconds'); axes[2].set_title('MFPT: Train Time'); axes[2].set_yscale('log')
plt.tight_layout()
plt.savefig('/kaggle/working/figures/mfpt_comparison.png')
plt.show()

In [ ]:
import shap
explainer = shap.TreeExplainer(xgb_model)
sv = explainer.shap_values(X_feat_te_sc)

plt.figure(figsize=(10, 8))
shap.summary_plot(sv, X_feat_te_sc, feature_names=feature_names, show=False)
plt.title('MFPT SHAP Importance')
plt.savefig('/kaggle/working/figures/mfpt_shap_summary.png', bbox_inches='tight')
plt.show()

plt.figure(figsize=(10, 6))
shap.summary_plot(sv, X_feat_te_sc, feature_names=feature_names, plot_type='bar', show=False)
plt.title('MFPT Mean |SHAP|')
plt.savefig('/kaggle/working/figures/mfpt_shap_bar.png', bbox_inches='tight')
plt.show()

ms = np.abs(sv).mean(0); total = ms.sum()
print("MFPT SHAP importance:")
for i in np.argsort(ms)[::-1]:
    print(f"  {feature_names[i]:>20s}: {ms[i]:.4f} ({ms[i]/total*100:.1f}%)")

In [ ]:
print("\n" + "=" * 60)
print("  N-CMAPSS vs MFPT COMPARISON")
print("=" * 60)

# N-CMAPSS results
nc = {'LR': 11.81, 'XGB': 9.26, 'CNN': 5.85, 'CNNLSTM': 5.81}
mf = {'LR': lr_rmse, 'XGB': xgb_rmse, 'CNN': cnn_rmse, 'CNNLSTM': cl_rmse}

print(f"\n{'Model':<20} {'N-CMAPSS':>10} {'MFPT':>10}")
print("-" * 42)
for key in ['LR', 'XGB', 'CNN', 'CNNLSTM']:
    print(f"{key:<20} {nc[key]:>10.2f} {mf[key]:>10.2f}")

nc_order = sorted(nc, key=lambda x: nc[x])
mf_order = sorted(mf, key=lambda x: mf[x])
print(f"\nN-CMAPSS ranking: {' < '.join(nc_order)}")
print(f"MFPT ranking:     {' < '.join(mf_order)}")
print(f"Rankings match: {nc_order == mf_order}")

In [ ]:
cnn.save('/kaggle/working/mfpt_cnn.keras')
cl.save('/kaggle/working/mfpt_cnn_lstm.keras')
xgb_model.save_model('/kaggle/working/mfpt_xgb.json')

print("\n=== MFPT RESULTS FOR REPORT ===\n")
print(f"Dataset: MFPT Bearing, {N_WINDOWS} windows from {len(recordings)} recordings")
print(f"Window: {WINDOW_SIZE} samples, step={STEP}, RUL cap={RUL_CAP}")
print(f"Features: {n_tab_features} hand-engineered (8 time-domain + 4 frequency-domain)")
print(f"Split: {split} train / {N_WINDOWS-split} test (temporal)")
print(f"\nLinear Regression: RMSE={lr_rmse:.2f}, MAE={lr_mae:.2f}, R²={lr_r2:.4f}")
print(f"XGBoost:           RMSE={xgb_rmse:.2f}, MAE={xgb_mae:.2f}, R²={xgb_r2:.4f}")
print(f"1D-CNN:            RMSE={cnn_rmse:.2f}, MAE={cnn_mae:.2f}, R²={cnn_r2:.4f}, params={cnn.count_params():,}")
print(f"CNN-LSTM:          RMSE={cl_rmse:.2f}, MAE={cl_mae:.2f}, R²={cl_r2:.4f}, params={cl.count_params():,}")
print(f"\nXGBoost vs LR:      {(1-xgb_rmse/lr_rmse)*100:+.1f}%")
print(f"1D-CNN vs XGBoost:  {(1-cnn_rmse/xgb_rmse)*100:+.1f}%")
print(f"CNN-LSTM vs XGBoost:{(1-cl_rmse/xgb_rmse)*100:+.1f}%")
print(f"\nFigures:")
for f in sorted(os.listdir('/kaggle/working/figures/')): print(f"  {f}")
print("\nDone.")